# RAG System - Interactive Query Interface

This notebook allows you to interactively query your PDF documents using the RAG pipeline.

## 1. Setup and Imports

In [ ]:
import os
import sys
import glob
from src.pdf_processor import process_pdf
from src.rag_pipeline import RAGPipeline
from config import CHUNK_SIZE, OVERLAP, PDF_DIR, OUTPUT_DIR, EMBEDDING_MODEL

print(f"Configuration:")
print(f"- PDF Directory: {PDF_DIR}")
print(f"- Embedding Model: {EMBEDDING_MODEL}")
print(f"- Chunk Size: {CHUNK_SIZE}")
print(f"- Overlap: {OVERLAP}")

## 2. Process PDFs and Build Index

In [ ]:
def process_all_pdfs(pdf_dir=PDF_DIR, chunk_size=CHUNK_SIZE, overlap=OVERLAP, output_dir=OUTPUT_DIR):
    pdf_files = glob.glob(os.path.join(pdf_dir, '*.pdf'))
    
    if not pdf_files:
        print(f"No PDF files found in {pdf_dir}")
        return []
    
    all_chunks = []
    for pdf_path in pdf_files:
        print(f"Processing: {os.path.basename(pdf_path)}")
        chunks = process_pdf(pdf_path, chunk_size, overlap, output_dir)
        all_chunks.extend(chunks)
    
    return all_chunks

# Process PDFs
print("Processing PDFs...")
chunks = process_all_pdfs()
print(f"Total chunks created: {len(chunks)}")

In [ ]:
# Initialize RAG Pipeline
print("Initializing RAG Pipeline...")
rag = RAGPipeline(model_name=EMBEDDING_MODEL)

# Build index
print("Building embeddings and FAISS index...")
chunks_with_embeddings = rag.build_index(chunks)
print(f"Index built successfully with {len(chunks_with_embeddings)} chunks")
print(f"Embedding dimension: {rag.embedding_model.get_dimension()}")

## 3. Interactive Query Interface

**Change the query below and run the cell to get relevant chunks:**

In [ ]:
# 🔍 CHANGE YOUR QUERY HERE
query = "What is Memory Caching?"
k = 5  # Number of results to return

# Search
print(f"Query: '{query}'")
print(f"Searching for top {k} relevant chunks...\n")

results = rag.query(query, k=k)

# Display results
print(f"Found {len(results)} results:\n")
print("=" * 80)

for i, result in enumerate(results, 1):
    chunk = result["chunk"]
    distance = result["distance"]
    
    print(f"\n🔸 Result {i} [Similarity Score: {1/(1+distance):.4f}]")
    print(f"📄 Source: {chunk['source']} (Page {chunk['page']})")
    print(f"🆔 Chunk ID: {chunk['chunk_id']}")
    print(f"📝 Text: {chunk['text'][:300]}{'...' if len(chunk['text']) > 300 else ''}")
    
    if chunk['images']:
        print(f"🖼️  Images: {len(chunk['images'])} image(s) found")
    
    print("-" * 80)

## 4. Quick Query Function

Use this cell for rapid testing with different queries:

In [ ]:
def quick_query(query_text, num_results=3):
    """Quick query function for testing"""
    results = rag.query(query_text, k=num_results)
    
    print(f"🔍 Query: '{query_text}'")
    print(f"📊 Top {len(results)} results:\n")
    
    for i, result in enumerate(results, 1):
        chunk = result["chunk"]
        score = 1/(1+result["distance"])
        
        print(f"{i}. [{score:.3f}] {chunk['source']} (Page {chunk['page']})")
        print(f"   {chunk['text'][:150]}...\n")

# Test different queries here:
quick_query("neural networks")
# quick_query("machine learning")
# quick_query("algorithms")

## 5. Advanced Search Options

In [ ]:
def advanced_search(query_text, k=5, min_score=0.5):
    """Advanced search with filtering"""
    results = rag.query(query_text, k=k)
    
    # Filter by minimum similarity score
    filtered_results = []
    for result in results:
        score = 1/(1+result["distance"])
        if score >= min_score:
            result["similarity_score"] = score
            filtered_results.append(result)
    
    print(f"🔍 Query: '{query_text}'")
    print(f"📊 Found {len(filtered_results)} results above {min_score} similarity threshold:\n")
    
    for i, result in enumerate(filtered_results, 1):
        chunk = result["chunk"]
        score = result["similarity_score"]
        
        print(f"🔸 Result {i} [Score: {score:.4f}]")
        print(f"📄 {chunk['source']} (Page {chunk['page']})")
        print(f"📝 {chunk['text'][:200]}...\n")
    
    return filtered_results

# Example usage:
# advanced_search("memory optimization", k=10, min_score=0.6)

## 6. LLM Answer Generation via Ollama (Mistral 7B Q4)

In [ ]:
import ollama
from config import OLLAMA_MODEL, OLLAMA_URL

client = ollama.Client(host=OLLAMA_URL)

def ask(query_text, k=5):
    results = rag.query(query_text, k=k)
    context = "\n\n".join(r['chunk']['text'] for r in results)
    prompt = f"Use the context below to answer the question.\n\nContext:\n{context}\n\nQuestion: {query_text}\nAnswer:"
    response = client.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return response['message']['content']

# 💬 Change your question here
answer = ask("What is Memory Caching?")
print(answer)

## 7. Save/Load Index (Optional)

In [ ]:
# Save the index for future use
index_path = "faiss_index.bin"
rag.save_index(index_path)
print(f"Index saved to {index_path}")

# To load later:
# rag.load_index(index_path, chunks)
# print("Index loaded successfully")